In [ ]:
"""
Kaggle notebook — multi-horizon AQI forecasting, LSTM (improved architecture +
hyperparameter search).

CHANGES:
  - build_model() is now parameterized: arbitrary stack of LSTM layers,
    per-layer dropout/recurrent_dropout, optional L2 weight decay,
    optional Bidirectional wrapping, configurable learning rate.
  - A small hyperparameter search (HP_CONFIGS below) trains several
    architecture/regularization variants and picks a winner using
    VALIDATION loss only — the holdout set is evaluated exactly once,
    on the winning config, to keep it a true holdout.
  - Results (val for all configs, holdout for the winner) are written to
    lstm_results.csv 

SETUP ON KAGGLE (one-time):
  1. Add-ons -> Secrets -> add a secret named HOPSWORKS_API_KEY with your key
  2. pip installs below run automatically in the first cell

Usage: run top to bottom as-is.
"""

# Install deps (Kaggle base image doesn't have hopsworks)
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hopsworks"], check=True)

# hopsworks pulls in protobuf<5, but Kaggle's preinstalled tensorflow needs
# protobuf>=5.28. If we `import tensorflow` now it will crash with
# "cannot import name 'runtime_version' from 'google.protobuf'". Force
# protobuf back up BEFORE importing tensorflow (nothing has imported it
# yet, so no kernel restart is needed) 

# Fixes the tf import without touching hopsworks' own functionality, which doesn't care about the
# protobuf version bump.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "protobuf>=5.28,<6"],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pyopenssl"],
    check=True,
)

import logging
import os
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers, regularizers

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("aqi_training")

# ── Constants ─────────────────────────────────────────────────────────────
FEATURE_GROUP_NAME = "aqi_features"
FEATURE_GROUP_VERSION = 1

TARGET_COLUMNS = ["target_aqi_24h", "target_aqi_48h", "target_aqi_72h"]
HORIZONS = [24, 48, 72]

LOOKBACK_HOURS = 72
GAP_INTERPOLATE_LIMIT = 2
HOLDOUT_FRAC = 0.15
VAL_FRAC = 0.10
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 10
LOSS_WEIGHTS = [1.0, 1.2, 1.5]
SEED = 42
RESULTS_CSV = "/kaggle/working/lstm_results.csv"

tf.random.set_seed(SEED)
np.random.seed(SEED)

# Hyperparameter / architecture search space 
# Each config fully describes one architecture variant.
# (each entry is a full train-to-convergence run under EarlyStopping).
HP_CONFIGS = [
    {
        "name": "baseline_64_32",
        "lstm_units": [64, 32],
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "l2": 0.0,
        "dense_units": 16,
        "learning_rate": 1e-3,
        "bidirectional": False,
    },
    {
        "name": "wider_128_64",
        "lstm_units": [128, 64],
        "dropout": 0.3,
        "recurrent_dropout": 0.1,
        "l2": 1e-5,
        "dense_units": 32,
        "learning_rate": 1e-3,
        "bidirectional": False,
    },
    {
        "name": "deeper_64_64_32",
        "lstm_units": [64, 64, 32],
        "dropout": 0.25,
        "recurrent_dropout": 0.0,
        "l2": 1e-5,
        "dense_units": 16,
        "learning_rate": 5e-4,
        "bidirectional": False,
    },
    {
        "name": "bidirectional_64_32",
        "lstm_units": [64, 32],
        "dropout": 0.2,
        "recurrent_dropout": 0.0,
        "l2": 0.0,
        "dense_units": 16,
        "learning_rate": 1e-3,
        "bidirectional": True,
    },
    {
        "name": "regularized_96_48_lowlr",
        "lstm_units": [96, 48],
        "dropout": 0.3,
        "recurrent_dropout": 0.1,
        "l2": 1e-4,
        "dense_units": 24,
        "learning_rate": 3e-4,
        "bidirectional": False,
    },
]


# Hopsworks connection 
def get_feature_store():
    import hopsworks

    api_key = None
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("HOPSWORKS_API_KEY")
    except Exception:
        api_key = os.environ.get("HOPSWORKS_API_KEY")

    if not api_key:
        raise RuntimeError(
            "No Hopsworks API key found. Add it under Add-ons -> Secrets as "
            "HOPSWORKS_API_KEY, or set the HOPSWORKS_API_KEY environment variable."
        )

    project = hopsworks.login(api_key_value=api_key)
    return project.get_feature_store()


def load_feature_group() -> pd.DataFrame:
    logger.info("Connecting to Hopsworks and reading '%s' v%d", FEATURE_GROUP_NAME, FEATURE_GROUP_VERSION)
    fs = get_feature_store()
    fg = fs.get_feature_group(FEATURE_GROUP_NAME, version=FEATURE_GROUP_VERSION)
    df = fg.read()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)
    logger.info("Loaded %d rows, %d columns from Hopsworks", *df.shape)
    return df


# Gap handling 
def reindex_to_hourly_grid(df: pd.DataFrame, interpolate_limit: int = 2) -> pd.DataFrame:
    df = df.sort_values("timestamp").reset_index(drop=True)
    full_range = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h", tz=df["timestamp"].dt.tz)
    df = df.set_index("timestamp").reindex(full_range).rename_axis("timestamp").reset_index()

    n_missing = df.drop(columns="timestamp").isna().any(axis=1).sum()
    logger.info("Reindexed to full hourly grid: %d rows, %d had at least one missing value", len(df), n_missing)

    non_ts_cols = [c for c in df.columns if c != "timestamp"]
    df[non_ts_cols] = df[non_ts_cols].interpolate(method="linear", limit=interpolate_limit, limit_area="inside")

    still_missing = df.drop(columns="timestamp").isna().any(axis=1).sum()
    if still_missing:
        logger.warning(
            "%d rows still NaN after interpolation (gaps longer than %d hours) — "
            "any window touching these will be dropped, not fabricated.",
            still_missing, interpolate_limit,
        )
    return df


def get_feature_cols(df: pd.DataFrame, target_cols: list[str]) -> list[str]:
    # "timestamp" is explicitly excluded — it is a datetime index used for
    # alignment only and must never reach the model as a feature.
    exclude = {"timestamp", "has_target"} | set(target_cols)
    feature_cols = [c for c in df.columns if c not in exclude]
    assert "timestamp" not in feature_cols, "timestamp leaked into feature columns"
    return feature_cols


# Gap aware seq building
def build_sequences(df, feature_cols, target_cols, lookback=LOOKBACK_HOURS):
    work = df.copy()
    if not work["timestamp"].is_monotonic_increasing:
        raise ValueError("df must be sorted by timestamp ascending")

    target_mask = work[target_cols].notna().all(axis=1)
    valid_idx = work.index[target_mask].to_list()

    feature_array = work[feature_cols].to_numpy(dtype=np.float32)
    target_array = work[target_cols].to_numpy(dtype=np.float32)
    timestamp_array = work["timestamp"].to_numpy()

    samples_X, samples_y, samples_ts = [], [], []
    dropped_gap, dropped_nan, dropped_short = 0, 0, 0

    for i in valid_idx:
        if i - lookback + 1 < 0:
            dropped_short += 1
            continue

        ts_window = timestamp_array[i - lookback + 1: i + 1]
        deltas = np.diff(ts_window).astype("timedelta64[h]")
        if not np.all(deltas == np.timedelta64(1, "h")):
            dropped_gap += 1
            continue

        window = feature_array[i - lookback + 1: i + 1]
        if np.isnan(window).any():
            dropped_nan += 1
            continue

        samples_X.append(window)
        samples_y.append(target_array[i])
        samples_ts.append(timestamp_array[i])

    logger.info(
        "Window filtering: %d kept, %d dropped (gap-spanning), %d dropped (NaN feature), %d dropped (short history)",
        len(samples_X), dropped_gap, dropped_nan, dropped_short,
    )

    if not samples_X:
        raise ValueError("No valid sequences could be built. Check lookback, NaN handling, gap frequency, and target alignment.")

    X = np.stack(samples_X, axis=0)
    y = np.stack(samples_y, axis=0)
    timestamps = pd.DatetimeIndex(samples_ts)
    logger.info("Built %d sequences | X %s | y %s | %s -> %s", len(X), X.shape, y.shape, timestamps[0], timestamps[-1])
    return X, y, timestamps

def chronological_split(X, y, timestamps):
    n = len(X)
    holdout_start = int(n * (1 - HOLDOUT_FRAC))
    val_start = int(holdout_start * (1 - VAL_FRAC))

    X_train, y_train = X[:val_start], y[:val_start]
    X_val, y_val = X[val_start:holdout_start], y[val_start:holdout_start]
    X_test, y_test = X[holdout_start:], y[holdout_start:]
    ts_test = timestamps[holdout_start:]

    logger.info("Split — train: %d  val: %d  holdout: %d", len(X_train), len(X_val), len(X_test))
    return X_train, y_train, X_val, y_val, X_test, y_test, ts_test


def scale(X_train, X_val, X_test):
    n_train, t, f = X_train.shape
    scaler = StandardScaler()
    X_train_2d = scaler.fit_transform(X_train.reshape(-1, f))
    X_val_2d = scaler.transform(X_val.reshape(-1, f))
    X_test_2d = scaler.transform(X_test.reshape(-1, f))
    return (
        X_train_2d.reshape(n_train, t, f),
        X_val_2d.reshape(len(X_val), t, f),
        X_test_2d.reshape(len(X_test), t, f),
        scaler,
    )


# ── Model (parameterized) ───────────────────────────────────────────────
def build_model(lookback, n_features, config: dict):
    """
    Build a stacked-LSTM multi-output model from a config dict:
      lstm_units:        list[int]  — one entry per LSTM layer, stacked
      dropout:            float     — Dropout after every LSTM layer
      recurrent_dropout:  float     — recurrent_dropout inside each LSTM
      l2:                 float     — L2 weight decay on LSTM + Dense kernels (0 disables)
      dense_units:        int       — width of the shared Dense layer before output heads
      learning_rate:       float
      bidirectional:       bool     — wrap every LSTM layer in Bidirectional
    """
    reg = regularizers.l2(config["l2"]) if config["l2"] > 0 else None
    inp = keras.Input(shape=(lookback, n_features), name="sequence_input")
    x = inp
    n_layers = len(config["lstm_units"])
    for i, units in enumerate(config["lstm_units"]):
        return_sequences = i < n_layers - 1
        lstm_layer = layers.LSTM(
            units,
            return_sequences=return_sequences,
            recurrent_dropout=config["recurrent_dropout"],
            kernel_regularizer=reg,
            name=f"lstm_{i + 1}",
        )
        if config["bidirectional"]:
            lstm_layer = layers.Bidirectional(lstm_layer, name=f"bilstm_{i + 1}")
        x = lstm_layer(x)
        x = layers.Dropout(config["dropout"], name=f"drop_{i + 1}")(x)

    x = layers.Dense(config["dense_units"], activation="relu", kernel_regularizer=reg, name="shared_dense")(x)

    out_24h = layers.Dense(1, name="out_24h")(x)
    out_48h = layers.Dense(1, name="out_48h")(x)
    out_72h = layers.Dense(1, name="out_72h")(x)

    model = keras.Model(inputs=inp, outputs=[out_24h, out_48h, out_72h], name=config["name"])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
        loss={"out_24h": "mse", "out_48h": "mse", "out_72h": "mse"},
        loss_weights={"out_24h": LOSS_WEIGHTS[0], "out_48h": LOSS_WEIGHTS[1], "out_72h": LOSS_WEIGHTS[2]},
        metrics={"out_24h": "mae", "out_48h": "mae", "out_72h": "mae"},
    )
    return model


def train(model, X_train, y_train, X_val, y_val, verbose=1):
    early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=0)
    return model.fit(
        X_train,
        [y_train[:, 0], y_train[:, 1], y_train[:, 2]],
        validation_data=(X_val, [y_val[:, 0], y_val[:, 1], y_val[:, 2]]),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=verbose,
    )


def regression_metrics(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {"rmse": float("nan"), "mae": float("nan"), "r2": float("nan")}
    return {
        "rmse": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 3),
        "mae": round(float(mean_absolute_error(y_true, y_pred)), 3),
        "r2": round(float(r2_score(y_true, y_pred)), 3),
    }


def evaluate(model, X, y) -> dict:
    preds = model.predict(X, verbose=0)
    return {h: regression_metrics(y[:, i], preds[i].squeeze()) for i, h in enumerate(HORIZONS)}


def append_results(rows: list[dict]):
    """Append rows to the shared results CSV so LSTM / classical / SHAP-pruned
    results can be joined later for champion-model selection."""
    new_df = pd.DataFrame(rows)
    if os.path.exists(RESULTS_CSV):
        existing = pd.read_csv(RESULTS_CSV)
        new_df = pd.concat([existing, new_df], ignore_index=True)
    new_df.to_csv(RESULTS_CSV, index=False)
    logger.info("Wrote %d rows to %s (%d total)", len(rows), RESULTS_CSV, len(new_df))


# Run 
df = load_feature_group()

missing = [c for c in TARGET_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(
        f"Expected target columns not found: {missing}. "
        f"Update TARGET_COLUMNS at the top of this script. "
        f"Available columns: {list(df.columns)}"
    )

df = reindex_to_hourly_grid(df, interpolate_limit=GAP_INTERPOLATE_LIMIT)
feature_cols = get_feature_cols(df, TARGET_COLUMNS)
logger.info("Using %d feature columns (timestamp excluded), %d targets", len(feature_cols), len(TARGET_COLUMNS))

X, y, timestamps = build_sequences(df, feature_cols, TARGET_COLUMNS, lookback=LOOKBACK_HOURS)
X_train, y_train, X_val, y_val, X_test, y_test, ts_test = chronological_split(X, y, timestamps)
X_train, X_val, X_test, scaler = scale(X_train, X_val, X_test)

n_features = X_train.shape[2]

# ── Hyperparameter / architecture search using VAL only ─────────
search_log = []
trained_models = {}

for config in HP_CONFIGS:
    logger.info("=== Training config: %s ===", config["name"])
    model = build_model(LOOKBACK_HOURS, n_features, config)
    history = train(model, X_train, y_train, X_val, y_val, verbose=0)
    epochs_run = len(history.history["loss"])
    best_val_loss = min(history.history["val_loss"])
    val_metrics = evaluate(model, X_val, y_val)

    logger.info(
        "    %s | epochs=%d | best_val_loss=%.3f | val RMSE 24h/48h/72h = %.2f/%.2f/%.2f",
        config["name"], epochs_run, best_val_loss,
        val_metrics[24]["rmse"], val_metrics[48]["rmse"], val_metrics[72]["rmse"],
    )

    search_log.append({"config": config["name"], "epochs_run": epochs_run, "best_val_loss": best_val_loss, "val_metrics": val_metrics})
    trained_models[config["name"]] = model

    for h in HORIZONS:
        append_results([{
            "model": f"LSTM_{config['name']}", "horizon": h, "split": "val",
            **val_metrics[h],
        }])

# ── Pick the winner by best (lowest) aggregate val_loss ────────────────────
winner_entry = min(search_log, key=lambda r: r["best_val_loss"])
winner_name = winner_entry["config"]
winner_model = trained_models[winner_name]
logger.info("Winning config: %s (best_val_loss=%.3f)", winner_name, winner_entry["best_val_loss"])

print("\n" + "=" * 70)
print("VALIDATION COMPARISON — all architecture/hyperparameter configs")
print("=" * 70)
for r in search_log:
    tag = " <== SELECTED" if r["config"] == winner_name else ""
    print(f"\n{r['config']}{tag}  (epochs run: {r['epochs_run']}, best_val_loss={r['best_val_loss']:.3f})")
    for h in HORIZONS:
        m = r["val_metrics"][h]
        print(f"  {h}h  RMSE={m['rmse']:.2f}  MAE={m['mae']:.2f}  R2={m['r2']:.3f}")
print("=" * 70)

# ── Holdout is touched on the winning model only ─────────────
holdout_results = evaluate(winner_model, X_test, y_test)

print("\n" + "=" * 60)
print(f"LSTM HOLDOUT RESULTS — winning config: {winner_name}")
print("(holdout, never seen during training, val-based selection, or early stopping)")
print("=" * 60)
for h in HORIZONS:
    r = holdout_results[h]
    print(f"  {h}h  RMSE={r['rmse']:.2f}  MAE={r['mae']:.2f}  R2={r['r2']:.3f}")
print("=" * 60)

for h in HORIZONS:
    append_results([{
        "model": f"LSTM_{winner_name}_WINNER", "horizon": h, "split": "holdout",
        **holdout_results[h],
    }])

winner_model.save(f"/kaggle/working/aqi_multi_horizon_lstm_{winner_name}.keras")
logger.info("Winning model saved to /kaggle/working/aqi_multi_horizon_lstm_%s.keras", winner_name)
logger.info("All val + holdout results appended to %s for cross-model comparison", RESULTS_CSV)